# Fig 4C / 4D / S9B / S9D (v2) — Meta-analysis of divergent populations, AFR + EUR (data processing)

v2 design (`33.Fig4C_4D_S9B_S9D.AFR_EUR_Meta_v2`): total n = 600, NCP = n × per-SNP h2g, AFR share 0–100%, in-sample LD from the sampled rows, references from the whole panels, and SuSiE-rRSS fits next to SuSiE-RSS. The v1 notebook and its `figure_data` are left as they are; everything here is written with a `_v2` name. The identity-LD fits are not used.

**Panel sets** (keys of `pal_meta_afr_eur`; the estimated mixture is shown as "Estimated LD mixture" by `meta_ld_label()`):

| | Fig 4 (`main`) | Supplement (`rrss`) |
|---|---|---|
| Estimated mixture | SuSiE-RSS, weights estimated then fixed (`manual_refpanel`, as v1) | SuSiE-rRSS (`mix_refpanel_rrss`) |
| Oracle | SuSiE-RSS on the in-sample LD (`gen_insample`) | same |
| AFR only / EUR only | SuSiE-RSS (`afr_refpanel`, `eur_refpanel`) | SuSiE-rRSS (`*_rrss`) |

The Oracle uses the LD of the sampled rows themselves, the exact generative LD of z_meta. The AFR cohort is drawn from the unrelated AFR panel, so even at AFR share 100% the in-sample LD differs from the AFR panel LD.

**SuSiE-rRSS is the final result:** for every `*_rrss` fit, the SER fallback model replaces the fit when `R_reliability_flag` is TRUE (the `<fit>_SER` rows of the summary).

**Inputs:** `AFR_EUR_meta_v2/summary/` (`[meta_v2_summary]`: the summary table, `_dropped`, `_ld_diff`)

**Outputs:**
- `figure_data/Fig4C_v2.AFR_EUR_Weight_Estimation` — estimated vs. true AFR proportion: SuSiE-RSS mixture (Fig 4) and SuSiE-rRSS mixture (supplement) (+ `_accuracy.csv`)
- `figure_data/Fig4D_v2.AFR_EUR_Mix_Power_FDR` — power / FDR (bootstrap 95% CI) of the two panel sets: `panel` = `main` (SuSiE-RSS) or `rrss` (SuSiE-rRSS); every fit in `Fig4D_v2.AFR_EUR_Mix_Power_FDR_All_Fits`
- `figure_data/FigS9_v2.AFR_EUR_SER_Fallback` — SER fallback rate of each SuSiE-rRSS fit
- `figure_data/FigS9_v2.AFR_EUR_LD_Difference` — mean LD difference of each reference from the in-sample LD
- `figure_data/FigS9D_v2.AFR_EUR_Hetero_Weight_Estimation` — proportion estimation when AFR and EUR have distinct causal variants (+ `_accuracy.csv`)

In [15]:
source("../../R/utils.R")

sum_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_meta_v2/summary"
res_file <- list.files(sum_dir, pattern = "_meta_v2_summary\\.rds$", full.names = TRUE)
stopifnot(length(res_file) == 1)
n_boot <- 1000
not_used <- c("identity", "identity_rrss")      # identity-LD fits

method_labels <- c(
  gen_insample      = "Oracle: in-sample LD (SuSiE-RSS)",
  gen_refpanel      = "Reference mixture, true weight (SuSiE-RSS)",
  manual_refpanel   = "Estimated mixture (SuSiE-RSS)",
  mix_refpanel      = "Estimated mixture, weight fitted jointly (SuSiE-RSS)",
  afr_refpanel      = "AFR only (SuSiE-RSS)",
  eur_refpanel      = "EUR only (SuSiE-RSS)",
  mix_refpanel_rrss = "Estimated mixture (SuSiE-rRSS)",
  afr_refpanel_rrss = "AFR only (SuSiE-rRSS)",
  eur_refpanel_rrss = "EUR only (SuSiE-rRSS)",
  susiex_insample   = "SuSiEx, in-sample LD",
  susiex_refpanel   = "SuSiEx, reference LD"
)
## panel sets: values are keys of pal_meta_afr_eur
main_labels <- c(
  manual_refpanel = "SuSiE-rRSS",      # palette key of the estimated mixture (shown as "Estimated LD mixture")
  gen_insample    = "Oracle",
  afr_refpanel    = "AFR only",
  eur_refpanel    = "EUR only"
)
rrss_labels <- c(
  mix_refpanel_rrss = "SuSiE-rRSS",
  gen_insample      = "Oracle",
  afr_refpanel_rrss = "AFR only",
  eur_refpanel_rrss = "EUR only"
)
stopifnot(setequal(main_labels, names(pal_meta_afr_eur)), setequal(rrss_labels, names(pal_meta_afr_eur)))
panels <- bind_rows(tibble(panel = "main", method = names(main_labels), label = unname(main_labels)),
                    tibble(panel = "rrss", method = names(rrss_labels), label = unname(rrss_labels)))

## bootstrap 95% CI of power / FDR over simulations; same estimates and resampling as
## boot_cell_mix() (R/utils.R), vectorized over the B resamples
boot_cell_fast <- function(d, B = 1000) {
  R   <- nrow(d)
  idx <- matrix(sample.int(R, R * B, replace = TRUE), nrow = R)
  bs  <- function(x) colSums(matrix(x[idx], nrow = R))
  tv <- bs(d$true_var_number); av <- bs(d$total_var_number)
  ts <- bs(d$true_set_number); as <- bs(d$total_set_number)
  fdn <- sum(d$total_set_number)
  qp <- .q(tv / av); qf <- .q(ifelse(as == 0, NA_real_, 1 - ts / as))
  tibble(R = R,
         power = sum(d$true_var_number) / sum(d$total_var_number), power_lo = qp[1], power_hi = qp[2],
         FDR   = if (fdn == 0) 0 else 1 - sum(d$true_set_number) / fdn, FDR_lo = qf[1], FDR_hi = qf[2])
}

## Simulations and dropped runs

In [16]:
res     <- readRDS(res_file) %>% filter(!str_remove(method, "_SER$") %in% not_used)
dropped <- readRDS(sub("\\.rds$", "_dropped.rds", res_file))

## simulations per setting, and the dropped ones (causal variant monomorphic in a sampled cohort, ...)
res %>% distinct(input_file, ncp, causal, prop) %>% count(ncp, causal, prop) %>%
  pivot_wider(names_from = prop, values_from = n, names_prefix = "AFR share ")
## an empty table (no columns) when no simulation was dropped
if (nrow(dropped)) count(dropped, reason, sort = TRUE) else cat("No simulation dropped\n")
res %>% filter(!fit_ok) %>% count(method, name = "n_failed_fits")
stopifnot(all(str_remove(unique(res$method), "_SER$") %in% names(method_labels)))

ncp,causal,AFR share 0.0,AFR share 0.2,AFR share 0.5,AFR share 0.8,AFR share 1.0
<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>
20,1,496,454,427,496,496
20,2,496,385,329,444,496
20,3,484,348,297,404,496
40,1,496,436,361,475,496
40,2,496,382,274,410,496
40,3,475,380,258,395,492
80,1,496,411,322,447,480
80,2,480,366,243,357,460
80,3,452,354,211,328,449


reason,n
<chr>,<int>
a causal variant is monomorphic in a sampled cohort,4


method,n_failed_fits
<chr>,<int>
afr_refpanel_rrss,1
gen_insample,974
mix_refpanel,13
mix_refpanel_rrss,21


## Final SuSiE-rRSS result and SER fallback

In [17]:
## final SuSiE-rRSS result: the SER model replaces a *_rrss fit when it is flagged
base <- res %>% filter(!str_ends(method, "_SER"))
ser  <- res %>% filter(str_ends(method, "_SER")) %>% mutate(method = str_remove(method, "_SER$"))
use_ser <- base %>%
  filter(str_ends(method, "_rrss"), R_reliability_flag %in% TRUE) %>%
  select(input_file, method) %>%
  semi_join(x = ser, by = c("input_file", "method"))
final <- base %>%
  anti_join(select(use_ser, input_file, method), by = c("input_file", "method")) %>%
  bind_rows(mutate(use_ser, ser_fallback = TRUE)) %>%
  mutate(ser_fallback = ser_fallback %in% TRUE,
         rrss         = str_ends(method, "_rrss"),
         weight       = as.numeric(prop),
         method_label = factor(unname(method_labels[method]), levels = unname(method_labels))) %>%
  filter(fit_ok)
stopifnot(!any(duplicated(select(final, input_file, method))))

## flagged *_rrss fits without a stored SER model keep the fit (should be none)
base %>% filter(str_ends(method, "_rrss"), R_reliability_flag %in% TRUE) %>%
  anti_join(use_ser, by = c("input_file", "method")) %>% count(method, name = "flagged_without_SER")

## SER fallback rate of each SuSiE-rRSS fit
fallback <- final %>% filter(rrss) %>%
  group_by(method, method_label, ncp, causal, weight) %>%
  summarise(n_sim = n(), n_flagged = sum(R_reliability_flag %in% TRUE),
            pct_fallback = 100 * mean(ser_fallback), .groups = "drop") %>%
  left_join(filter(panels, panel == "rrss") %>% select(method, label), by = "method")
fallback %>% group_by(method_label, ncp) %>%
  summarise(pct_fallback = round(100 * sum(n_flagged) / sum(n_sim), 1), .groups = "drop") %>%
  pivot_wider(names_from = ncp, values_from = pct_fallback, names_prefix = "NCP ")
save_data(fallback, "FigS9_v2.AFR_EUR_SER_Fallback")

method,flagged_without_SER
<chr>,<int>


method_label,NCP 20,NCP 40,NCP 80,NCP 120
<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Estimated mixture (SuSiE-rRSS),0.0,0.0,0.1,0.1
AFR only (SuSiE-rRSS),2.3,4.9,8.9,19.5
EUR only (SuSiE-rRSS),54.7,51.6,47.8,48.2


## Fig 4C (v2): weight estimation

In [18]:
## one row per simulation: estimated AFR weight (omega[1]) of the SuSiE-RSS (Fig 4) and SuSiE-rRSS (supplement) mixtures
om <- res %>%
  filter(method == "mix_refpanel") %>%
  distinct(input_file, ncp, causal, prop, w, omega_1, omega_rrss_1) %>%
  pivot_longer(c(omega_1, omega_rrss_1), names_to = "model", values_to = "omega") %>%
  filter(!is.na(omega)) %>%
  mutate(model  = factor(recode(model, omega_1 = "SuSiE-RSS", omega_rrss_1 = "SuSiE-rRSS"), levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         weight = as.numeric(prop))
stopifnot(all(abs(om$w - om$weight) < 1e-6))      # realized share = design share

om_acc <- om %>%
  group_by(model, ncp, causal, weight) %>%
  summarise(mean_omega = mean(omega), sd_omega = sd(omega),
            MAE = mean(abs(omega - weight)), n = n(), .groups = "drop")
om_acc %>% group_by(model, weight) %>%
  summarise(MAE = round(mean(MAE), 3), n = sum(n), .groups = "drop") %>%
  pivot_wider(names_from = model, values_from = c(MAE, n))
write_csv(om_acc, file.path(data_dir, "Fig4C_v2.AFR_EUR_Weight_Estimation_accuracy.csv"))
save_data(om, "Fig4C_v2.AFR_EUR_Weight_Estimation")

weight,MAE_SuSiE-RSS,MAE_SuSiE-rRSS,n_SuSiE-RSS,n_SuSiE-rRSS
<dbl>,<dbl>,<dbl>,<int>,<int>
0.0,0.001,0.001,5743,5746
0.2,0.047,0.047,4609,4606
0.5,0.020,0.019,3427,3423
0.8,0.052,0.051,4746,4742
1.0,0.001,0.001,5669,5669


## Fig 4D (v2): power / FDR

In [19]:
## bootstrap 95% CIs for every fit x setting
set.seed(1)
summary_ci <- final %>%
  group_by(method, method_label, rrss, ncp, causal, weight) %>%
  group_modify(~ boot_cell_fast(.x, B = n_boot)) %>%
  ungroup()

w_levels <- sort(unique(summary_ci$weight))
to_long <- function(df) df %>%
  pivot_longer(c(power, power_lo, power_hi, FDR, FDR_lo, FDR_hi), names_to = "key", values_to = "value") %>%
  mutate(metric = factor(if_else(str_starts(key, "power"), "Power", "FDR"), levels = c("Power", "FDR")),
         key    = case_when(str_ends(key, "_lo") ~ "lo", str_ends(key, "_hi") ~ "hi", TRUE ~ "est")) %>%
  pivot_wider(names_from = key, values_from = value) %>%
  mutate(ncp_num = ncp,
         ncp     = fct_reorder(paste0("NCP = ", ncp_num), ncp_num),
         causal  = factor(paste0(causal, " causal"), levels = paste0(sort(unique(as.integer(causal))), " causal")),
         weight  = factor(weight, levels = w_levels, labels = paste0(round(w_levels * 100), "%")))

all_fits <- to_long(summary_ci)
## the two panel sets (the Oracle is in both)
pl <- summary_ci %>% inner_join(panels, by = "method", relationship = "many-to-many") %>% to_long() %>%
  mutate(method_key = method, method = factor(label, levels = names(pal_meta_afr_eur)), panel = factor(panel, levels = c("main", "rrss")))
stopifnot(!any(is.na(pl$method)))

## panel sets averaged over settings (quick check)
pl %>% group_by(panel, method, metric) %>% summarise(mean_est = round(mean(est, na.rm = TRUE), 3), .groups = "drop") %>%
  pivot_wider(names_from = metric, values_from = mean_est)
save_data(pl, "Fig4D_v2.AFR_EUR_Mix_Power_FDR")
save_data(all_fits, "Fig4D_v2.AFR_EUR_Mix_Power_FDR_All_Fits")

panel,method,Power,FDR
<fct>,<fct>,<dbl>,<dbl>
main,SuSiE-rRSS,0.836,0.018
main,Oracle,0.836,0.015
main,AFR only,0.711,0.362
main,EUR only,0.753,0.426
rrss,SuSiE-rRSS,0.827,0.015
rrss,Oracle,0.836,0.015
rrss,AFR only,0.699,0.178
rrss,EUR only,0.628,0.023


## LD difference from the in-sample LD

In [20]:
## LD difference of each reference from the in-sample LD, per simulation; the estimated mixtures depend on z, so NCP is kept
ld_diff <- readRDS(sub("\\.rds$", "_ld_diff.rds", res_file)) %>% filter(!reference %in% not_used)
ld_metrics <- intersect(c("mse", "max_abs_diff", "concord"), names(ld_diff))
ld_summ <- ld_diff %>%
  mutate(weight = as.numeric(prop)) %>%
  group_by(across(any_of(c("reference", "pairs"))), ncp, weight) %>%
  summarise(n_sim = n(), across(all_of(ld_metrics), ~ mean(.x, na.rm = TRUE)), .groups = "drop")
ld_summ %>% group_by(across(any_of(c("reference", "pairs")))) %>%
  summarise(across(all_of(ld_metrics), ~ signif(mean(.x, na.rm = TRUE), 3)), .groups = "drop")
save_data(ld_summ, "FigS9_v2.AFR_EUR_LD_Difference")

reference,pairs,mse,max_abs_diff,concord
<chr>,<chr>,<dbl>,<dbl>,<dbl>
afr_refpanel,all,0.008960,0.635,0.829
afr_refpanel,insample_r_gt_0.05,0.015100,0.632,0.860
eur_refpanel,all,0.008990,0.639,0.855
eur_refpanel,insample_r_gt_0.05,0.014400,0.636,0.882
gen_refpanel,all,0.000659,0.137,0.983
gen_refpanel,insample_r_gt_0.05,0.000841,0.133,0.991
mix_refpanel,all,0.000700,0.138,0.982
mix_refpanel,insample_r_gt_0.05,0.000916,0.134,0.990
mix_refpanel_rrss,all,0.000698,0.138,0.982


## Fig S9D (v2): heterogeneous causal variants

In [21]:
## distinct causal variants in AFR and EUR (33 [afr_eur_hetero_v2]): weight estimates only
library(parallel)
het_root  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_hetero_v2/result"
het_files <- list.files(het_root, pattern = "_hetero\\.rds$", full.names = TRUE, recursive = TRUE)
read_het <- function(f) tryCatch({
  r <- readRDS(f)
  if (!identical(r$status, "ok")) return(NULL)
  tibble(ncp = r$ncp, causal = r$causal, prop = r$prop, w = r$w, index = r$index,
         omega_1 = r$est_afr_prop %||% NA_real_, omega_rrss_1 = r$est_afr_prop_rrss %||% NA_real_)
}, error = function(e) { message("FAILED: ", f, " : ", conditionMessage(e)); NULL })
het <- bind_rows(mclapply(het_files, read_het, mc.cores = 4))
cat(length(het_files), "files,", nrow(het), "ok\n")

om_het <- het %>%
  pivot_longer(c(omega_1, omega_rrss_1), names_to = "model", values_to = "omega") %>%
  filter(!is.na(omega)) %>%
  mutate(model  = factor(recode(model, omega_1 = "SuSiE-RSS", omega_rrss_1 = "SuSiE-rRSS"), levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         weight = as.numeric(prop))
om_het_acc <- om_het %>%
  group_by(model, ncp, causal, weight) %>%
  summarise(mean_omega = mean(omega), sd_omega = sd(omega),
            MAE = mean(abs(omega - weight)), n = n(), .groups = "drop")
write_csv(om_het_acc, file.path(data_dir, "FigS9D_v2.AFR_EUR_Hetero_Weight_Estimation_accuracy.csv"))
save_data(om_het, "FigS9D_v2.AFR_EUR_Hetero_Weight_Estimation")

16678 files, 16670 ok
